# Task 8 — User-Based Collaborative Filtering

**Objective:** build a simple user-based collaborative filtering model from the interaction CSV created in Task 7.

> The interaction file is **simulated practice data** because the supplied talks dataset contains no actual user ratings. Results demonstrate the algorithm only; they are not evidence of real audience preferences.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from pathlib import Path
import pandas as pd


def find_project_root():
    """Find the project folder from the current Jupyter working directory."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw" / "ted_talks_merged.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/raw/ted_talks_merged.csv. "
        "Open Jupyter from the TED_Talks_Recommendation_Ajay project folder."
    )


PROJECT_DIR = find_project_root()
RAW_DATA_PATH = PROJECT_DIR / "data" / "raw" / "ted_talks_merged.csv"
PROCESSED_DATA_PATH = PROJECT_DIR / "data" / "processed" / "ted_talks_cleaned.csv"
print("Project folder:", PROJECT_DIR)

INTERACTION_PATH = PROJECT_DIR / "data" / "interaction" / "user_interactions.csv"
if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError("Run Task 3 first.")
if not INTERACTION_PATH.exists():
    raise FileNotFoundError("Run Task 7 first to create data/interaction/user_interactions.csv")

talks = pd.read_csv(PROCESSED_DATA_PATH)
interactions = pd.read_csv(INTERACTION_PATH)
required_interaction_columns = ["user_id", "talk_id", "rating"]
missing = [column for column in required_interaction_columns if column not in interactions.columns]
if missing:
    raise ValueError(f"Interaction dataset is missing: {missing}")
print("Interaction data shape:", interactions.shape)
display(interactions.head())

Project folder: /mnt/data/TED_Talks_Recommendation_Ajay


Interaction data shape: (810, 6)


,user_id,talk_id,rating,liked,watched,simulated_interest_group
0,user_01,Meaghan Ramsey: Why thinking you're ugly is ba...,3,False,True,technology
1,user_01,"Silk Road Ensemble: ""Turceasca""",3,False,True,technology
2,user_01,Steve Ramirez and Xu Liu: A mouse. A laser bea...,3,False,True,technology
3,user_01,Morgana Bailey: The danger of hiding who you are,4,True,True,technology
4,user_01,Erik Hersman: Reporting crisis via texting,4,True,True,technology


## 1. Create the user-item matrix

Rows are users; columns are TED Talks; values are ratings. We fill missing cells with zero for this first simple cosine-similarity implementation. In this notebook, zero means **no recorded rating**, not a real rating of zero.

In [3]:
user_item_matrix = interactions.pivot_table(
    index="user_id",
    columns="talk_id",
    values="rating",
    aggfunc="mean",
).fillna(0)

print("User-item matrix shape:", user_item_matrix.shape)
display(user_item_matrix.iloc[:8, :10])

User-item matrix shape: (18, 100)


talk_id,Adam Savage: My obsession with objects and the stories they tell,"Aicha el-Wafi + Phyllis Rodriguez: The mothers who found forgiveness, friendship",Alex Steffen: The route to a sustainable future,Amber Case: We are all cyborgs now,Ameera Harouda: Why I put myself in danger to tell the stories of Gaza,Anders Fjellberg: Two nameless bodies washed up on the beach. Here are their stories,"Andrew Solomon: Love, no matter what",Andrew Stanton: The clues to a great story,Andrew Youn: 3 reasons why we can win the fight against poverty,Anna Mracek Dietrich: A plane you can drive
user_id,,,,,,,,,,
user_01,3.0,0.0,0.0,3.0,0.0,0.0,0.0,3.0,0.0,0.0
user_02,0.0,3.0,0.0,0.0,3.0,3.0,3.0,0.0,0.0,0.0
user_03,0.0,5.0,0.0,0.0,4.0,4.0,0.0,5.0,4.0,0.0
user_04,0.0,0.0,0.0,3.0,0.0,2.0,0.0,4.0,0.0,0.0
user_05,4.0,0.0,0.0,2.0,0.0,3.0,0.0,3.0,0.0,0.0
user_06,4.0,0.0,0.0,0.0,0.0,2.0,3.0,4.0,0.0,3.0
user_07,2.0,3.0,4.0,0.0,0.0,0.0,3.0,3.0,2.0,4.0
user_08,3.0,3.0,3.0,0.0,3.0,3.0,2.0,0.0,4.0,0.0


## 2. Calculate user-user cosine similarity

In [4]:
user_similarity_values = cosine_similarity(user_item_matrix)
user_similarity = pd.DataFrame(
    user_similarity_values,
    index=user_item_matrix.index,
    columns=user_item_matrix.index,
)
print("User similarity matrix shape:", user_similarity.shape)
display(user_similarity.round(3))

User similarity matrix shape: (18, 18)


user_id,user_01,user_02,user_03,user_04,user_05,user_06,user_07,user_08,user_09,user_10,user_11,user_12,user_13,user_14,user_15,user_16,user_17,user_18
user_id,,,,,,,,,,,,,,,,,,
user_01,1.000,0.303,0.518,0.458,0.490,0.449,0.396,0.526,0.475,0.418,0.497,0.435,0.457,0.379,0.441,0.485,0.335,0.482
user_02,0.303,1.000,0.442,0.328,0.491,0.432,0.379,0.443,0.469,0.303,0.381,0.476,0.353,0.493,0.358,0.329,0.558,0.306
user_03,0.518,0.442,1.000,0.311,0.442,0.428,0.389,0.547,0.464,0.278,0.377,0.404,0.376,0.519,0.447,0.434,0.370,0.474
user_04,0.458,0.328,0.311,1.000,0.387,0.431,0.378,0.350,0.441,0.399,0.438,0.463,0.524,0.394,0.448,0.502,0.443,0.444
user_05,0.490,0.491,0.442,0.387,1.000,0.492,0.356,0.420,0.496,0.462,0.376,0.417,0.398,0.345,0.499,0.400,0.392,0.435
user_06,0.449,0.432,0.428,0.431,0.492,1.000,0.437,0.485,0.481,0.508,0.456,0.468,0.490,0.404,0.476,0.439,0.410,0.344
user_07,0.396,0.379,0.389,0.378,0.356,0.437,1.000,0.481,0.316,0.494,0.489,0.359,0.381,0.468,0.364,0.463,0.532,0.488
user_08,0.526,0.443,0.547,0.350,0.420,0.485,0.481,1.000,0.471,0.379,0.504,0.338,0.377,0.496,0.385,0.409,0.451,0.394
user_09,0.475,0.469,0.464,0.441,0.496,0.481,0.316,0.471,1.000,0.458,0.491,0.380,0.556,0.344,0.492,0.435,0.410,0.525


## 3. Find the most similar users

In [5]:
def similar_users(user_id, top_n=5):
    if user_id not in user_similarity.index:
        raise ValueError(f"Unknown user_id: {user_id}")
    scores = user_similarity.loc[user_id].drop(index=user_id)
    return scores.sort_values(ascending=False).head(top_n).rename("similarity_score").to_frame()

example_user = user_item_matrix.index[0]
print("Similar users for:", example_user)
display(similar_users(example_user, top_n=5))

Similar users for: user_01


,similarity_score
user_id,
user_08,0.525778
user_03,0.517555
user_11,0.497318
user_05,0.489961
user_16,0.484788


## 4. Build the recommendation function

For each talk the user has not rated, calculate a similarity-weighted average of ratings from the most similar users who have rated that talk. Talks already rated by the selected user are excluded.

In [6]:
def recommend_for_user(user_id, top_n=5, n_neighbors=5):
    if user_id not in user_item_matrix.index:
        raise ValueError(f"Unknown user_id: {user_id}. This user has no interaction history in the practice dataset.")
    if not isinstance(top_n, int) or top_n < 1:
        raise ValueError("top_n must be a positive integer.")
    if not isinstance(n_neighbors, int) or n_neighbors < 1:
        raise ValueError("n_neighbors must be a positive integer.")

    user_ratings = user_item_matrix.loc[user_id]
    rated_talk_ids = set(user_ratings[user_ratings > 0].index)
    neighbor_scores = user_similarity.loc[user_id].drop(index=user_id).sort_values(ascending=False).head(n_neighbors)

    recommendations = []
    for talk_id in user_item_matrix.columns:
        if talk_id in rated_talk_ids:
            continue
        neighbor_ratings = user_item_matrix.loc[neighbor_scores.index, talk_id]
        has_rating = neighbor_ratings > 0
        if not has_rating.any():
            continue
        weights = neighbor_scores.loc[has_rating.index[has_rating]]
        ratings = neighbor_ratings.loc[has_rating]
        total_weight = float(weights.sum())
        if total_weight <= 0:
            continue
        predicted_rating = float((ratings * weights).sum() / total_weight)
        recommendations.append({"talk_id": talk_id, "predicted_rating": predicted_rating})

    result = pd.DataFrame(recommendations)
    if result.empty:
        return result
    result = result.sort_values("predicted_rating", ascending=False).head(top_n)
    result = result.merge(talks[["talk_id", "title", "url"]].drop_duplicates("talk_id"), on="talk_id", how="left")
    return result[["talk_id", "title", "predicted_rating", "url"]].reset_index(drop=True)

## 5. Test at least three users

In [7]:
test_users = user_item_matrix.index[:3].tolist()
for user_id in test_users:
    print("\nRecommendations for:", user_id)
    display(recommend_for_user(user_id, top_n=5, n_neighbors=5))


Recommendations for: user_01


,talk_id,title,predicted_rating,url
0,James Balog: Time-lapse proof of extreme ice loss,Time-lapse proof of extreme ice loss,5.000000,https://www.ted.com/talks/james_balog_time_lap...
1,Fawn Qiu: Easy DIY projects for kid engineers,Easy DIY projects for kid engineers,5.000000,https://www.ted.com/talks/fawn_qiu_easy_diy_pr...
2,Conrad Wolfram: Teaching kids real math with c...,Teaching kids real math with computers,5.000000,https://www.ted.com/talks/conrad_wolfram_teach...
3,Jessa Gamble: Our natural sleep cycle is nothi...,Our natural sleep cycle is nothing like what w...,4.496059,https://www.ted.com/talks/jessa_gamble_how_to_...
4,Emiliano Salinas: A civil response to violence,A civil response to violence,4.009991,https://www.ted.com/talks/emiliano_salinas_a_c...



Recommendations for: user_02


,talk_id,title,predicted_rating,url
0,Michael Pawlyn: Using nature's genius in archi...,Using nature's genius in architecture,5.00000,https://www.ted.com/talks/michael_pawlyn_using...
1,Seth Berkley: HIV and flu -- the vaccine strategy,HIV and flu -- the vaccine strategy,4.00000,https://www.ted.com/talks/seth_berkley_hiv_and...
2,Jim Holt: Why does the universe exist?,Why does the universe exist?,4.00000,https://www.ted.com/talks/jim_holt_why_does_th...
3,Stefan Sagmeister: The power of time off,The power of time off,4.00000,https://www.ted.com/talks/stefan_sagmeister_th...
4,Kevin Bales: How to combat modern slavery,How to combat modern slavery,3.97647,https://www.ted.com/talks/kevin_bales_how_to_c...



Recommendations for: user_03


,talk_id,title,predicted_rating,url
0,Bill Davenhall: Your health depends on where y...,Your health depends on where you live,4.030714,https://www.ted.com/talks/bill_davenhall_your_...
1,Steve Ramirez and Xu Liu: A mouse. A laser bea...,A mouse. A laser beam. A manipulated memory.,4.027406,https://www.ted.com/talks/steve_ramirez_and_xu...
2,Uri Hasson: This is your brain on communication,This is your brain on communication,4.000000,https://www.ted.com/talks/uri_hasson_this_is_y...
3,Kevin Bales: How to combat modern slavery,How to combat modern slavery,3.962889,https://www.ted.com/talks/kevin_bales_how_to_c...
4,Jamila Raqib: The secret to effective nonviole...,The secret to effective nonviolent resistance,3.917678,https://www.ted.com/talks/jamila_raqib_the_sec...


## 6. Verify that already-rated talks are excluded

In [8]:
check_user = test_users[0]
recommended = recommend_for_user(check_user, top_n=10, n_neighbors=5)
already_rated = set(user_item_matrix.loc[check_user][user_item_matrix.loc[check_user] > 0].index)
recommended_ids = set(recommended["talk_id"]) if not recommended.empty else set()
print("Any already-rated talk recommended?", bool(already_rated & recommended_ids))
print("Recommendations sorted by predicted rating?", bool(recommended["predicted_rating"].is_monotonic_decreasing) if not recommended.empty else "No recommendations available")

Any already-rated talk recommended?

 False
Recommendations sorted by predicted rating? True


## 7. Review questions

1. **What is collaborative filtering?** Recommending items using patterns in interactions from users and items.
2. **How is it different from content-based filtering?** Content-based filtering compares item attributes; collaborative filtering uses user-item interaction patterns.
3. **What is a user-item matrix?** A table with users as rows, items as columns, and ratings/interactions as values.
4. **Why calculate user similarity?** To find users whose recorded preferences are alike.
5. **What does cosine similarity represent?** The cosine of the angle between two vectors; a larger positive value indicates more similar rating patterns in this representation.
6. **Why exclude already-rated talks?** The purpose is to suggest something the user has not already rated.
7. **What is a predicted rating?** An estimated preference score derived from similar users' ratings.
8. **What if a new user has no interaction history?** User-based collaborative filtering cannot identify similar users for them; ask for interests or use a content/popularity fallback.
9. **What is the cold-start problem?** The difficulty of recommending for a new user or new item with little or no interaction data.
10. **What are the limitations?** Sparse data, cold start, popularity bias, scalability, sensitivity to simulated/noisy ratings, and zero-filling's assumption that missing means no preference signal.

## 8. Important limitation

Zero-filling is used here only because the task asks for a simple first implementation. A missing interaction is not the same as a dislike or a zero rating. A more advanced model would distinguish observed ratings from missing values and evaluate recommendations using held-out real interactions.